# (부록) FastMCP 로 MCP Host · Client · Server 구축

**MCP(Model Context Protocol)** 는 LLM 앱(Host)이 외부 도구·데이터(Server)에 표준 방식으로
접속하기 위한 개방 프로토콜입니다. 구조는 **Host → Client(s) → Server(s)** 3계층입니다.

| 계층 | 역할 | 이 노트북에서 |
|---|---|---|
| **Server** | 도구(tool)·리소스·프롬프트를 제공 | `FastMCP` 서버 + `@mcp.tool` |
| **Client** | 서버에 연결해 도구 목록 조회·호출 | `fastmcp.Client` (인메모리 전송) |
| **Host** | LLM 이 어떤 도구를 쓸지 결정하고 Client 로 실행 | 로컬 **Ollama `qwen3:8b`** |

[FastMCP](https://gofastmcp.com/getting-started/welcome) 는 MCP 서버/클라이언트를 파이썬으로 빠르게
만들게 해 줍니다. 특히 **인메모리 전송**(Client 에 서버 객체를 직접 전달)을 지원해, 별도 프로세스나
네트워크 없이 노트북 한 곳에서 Server↔Client↔Host 전체 흐름을 실행할 수 있습니다.

> 📦 설치·실행: [`env_guides/M02_4_fastmcp.md`](env_guides/M02_4_fastmcp.md) · 공통 도구는 [`agentic_lib/`](agentic_lib)

### 전제 조건
- `M02_1_local_llm` 완료(로컬 Ollama + `qwen3:8b` 준비, `ollama serve`)
- 패키지 `fastmcp`(아래 셀에서 자동 설치)

---
## 0. 환경 설정

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()  # .env 재로드 + 현재 공급자 상태 출력

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text  # 공급자 무관 응답 정규화(<think> 제거)

# FastMCP 설치(이미 있으면 빠르게 통과)
utils.uv_install(['fastmcp'])

llm = utils.get_llm()
print('준비 완료:', utils.LLM_PROVIDER)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['fastmcp']


준비 완료: nvidia


---
## 1. MCP **Server** — `FastMCP` + `@mcp.tool`

`FastMCP(name=...)` 로 서버를 만들고, 일반 파이썬 함수에 `@mcp.tool` 를 붙이면 MCP 도구가 됩니다.
함수의 **타입 힌트와 docstring** 이 그대로 도구의 입력 스키마·설명이 되어 클라이언트/LLM 에게 노출됩니다.

In [2]:
import math, datetime
from fastmcp import FastMCP

# MCP 서버 생성
mcp = FastMCP(name="ToolServer", instructions="계산·날씨·시간 도구를 제공하는 MCP 서버")

# eval 에 노출할 안전한 math 심볼(임의 코드 실행 차단)
_MATH = {k: getattr(math, k) for k in ['sqrt','pow','sin','cos','tan','log','log10','exp','pi','e','factorial','floor','ceil']}

@mcp.tool
def calculator(expression: str) -> str:
    """수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'."""
    try:
        # LLM 이 자주 쓰는 캐럿(^)을 파이썬 거듭제곱(**)으로 보정
        return f"{expression} = {eval(expression.replace('^', '**'), {'__builtins__': {}}, _MATH)}"
    except Exception as e:
        return f"오류: {e}"

@mcp.tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다(시뮬레이션)."""
    return f"{city} 날씨: 맑음, 22도"

@mcp.tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print('MCP 서버 생성 완료:', mcp.name)

MCP 서버 생성 완료: ToolServer


---
## 2. MCP **Client** — 인메모리 전송으로 서버에 연결

`Client(mcp)` 처럼 **서버 객체를 직접 전달**하면 FastMCP 가 별도 프로세스/네트워크 없이 메모리 안에서
연결합니다(테스트·학습에 최적). 클라이언트는 `async` API 이므로 `async with` 로 연결 수명주기를 관리하고
`await` 로 호출합니다. (Jupyter 는 셀에서 최상위 `await` 를 지원합니다.)

- `await client.list_tools()` — 서버가 제공하는 도구 목록·스키마 조회
- `await client.call_tool(name, args)` — 도구 실행 → `CallToolResult`(`.data` 구조화 결과, `.content` 블록)

In [3]:
from fastmcp import Client

# 인메모리 전송: 서버 객체를 그대로 Client 에 전달
async with Client(mcp) as client:
    # 1) 도구 목록(디스커버리)
    tools = await client.list_tools()
    print('=== 서버가 제공하는 MCP 도구 ===')
    for t in tools:
        params = list((t.inputSchema or {}).get('properties', {}).keys())
        print(f"  - {t.name}{tuple(params)}: {(t.description or '').splitlines()[0]}")

    # 2) 도구 직접 호출
    print('\n=== 도구 호출 ===')
    r1 = await client.call_tool('calculator', {'expression': '2 ** 10'})
    print('  calculator ->', r1.data)          # .data: 구조화된 반환값
    r2 = await client.call_tool('get_weather', {'city': '서울'})
    print('  get_weather ->', r2.data)

=== 서버가 제공하는 MCP 도구 ===
  - calculator('expression',): 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.
  - get_weather('city',): 도시의 현재 날씨를 조회합니다(시뮬레이션).
  - get_current_time(): 현재 날짜와 시간을 반환합니다.

=== 도구 호출 ===
  calculator -> 2 ** 10 = 1024
  get_weather -> 서울 날씨: 맑음, 22도


---
## 3. MCP **Host** — LLM 이 도구를 선택하고 Client 로 실행

Host 는 LLM 앱입니다. 흐름은 다음과 같습니다.

```
사용자 요청 → [Host] LLM 이 MCP 도구 스키마를 보고 어떤 도구를 호출할지 결정(tool_calls)
           → [Client] 결정된 도구를 MCP 서버에 호출(call_tool)
           → [Server] 도구 실행 결과 반환
           → [Host] 결과를 LLM 에 되먹여 최종 자연어 답변 생성
```

클라이언트가 조회한 MCP 도구 스키마(`inputSchema`)를 그대로 LLM 의 `bind_tools()` 에 넘겨,
LLM 이 표준 `tool_calls` 로 도구를 선택하게 합니다.

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

async def mcp_host(question: str, max_turns: int = 8):
    """MCP Host 루프: LLM 의 도구 선택 → Client 로 서버 도구 실행 → 결과 되먹여 최종 답변.

    공급자에 따라 한 응답에 도구를 여러 개(병렬) 또는 하나씩(순차) 호출한다. NVIDIA build 의
    `llama-3.1-8b` 처럼 '한 번에 도구 하나만' 지원하는 서버는 대화 이력에 다중 tool_calls 가
    들어가면 500 오류('only supports single tool-calls')를 내므로, bootstrap.bind_tools 로
    단일 호출을 유도하고 bootstrap.cap_tool_calls 로 첫 호출만 남긴 뒤, 도구가 남으면 루프를
    돌며 순차적으로 처리한다(병렬 지원 공급자에는 아무 영향 없음).
    """
    async with Client(mcp) as client:
        # 1) 서버 도구를 조회해 LLM 용 스키마(OpenAI 함수 형식)로 변환
        mcp_tools = await client.list_tools()
        schemas = [{"type": "function",
                    "function": {"name": t.name, "description": t.description or "",
                                 "parameters": t.inputSchema}} for t in mcp_tools]
        # 공급자 차이 흡수: 단일 도구 호출 서버에는 parallel_tool_calls=False 유도
        llm_with_tools = bootstrap.bind_tools(llm, schemas)

        messages = [SystemMessage(content="필요하면 제공된 MCP 도구를 사용해 정확히 답하세요. /no_think"),
                    HumanMessage(content=question)]
        print(f"[질문] {question}")

        # 2~4) 도구 호출이 없어질 때까지: LLM 결정 → Client 로 실행 → 결과 되먹임
        resp = None
        for _ in range(max_turns):
            resp = llm_with_tools.invoke(messages)
            resp = bootstrap.cap_tool_calls(resp)   # 단일 도구 서버면 첫 호출만 남김(그 외 원본)
            messages.append(resp)
            if not resp.tool_calls:
                break                                # 더 호출할 도구 없음 → 최종 답변
            print(f"[Host→LLM 결정] tool_calls: {[(tc['name'], tc['args']) for tc in resp.tool_calls]}")
            for tc in resp.tool_calls:
                result = await client.call_tool(tc['name'], tc['args'])
                out = result.data if result.data is not None else result.content
                print(f"[Client→Server] {tc['name']}({tc['args']}) → {out}")
                messages.append(ToolMessage(content=str(out), tool_call_id=tc['id']))

        print(f"\n[최종 답변] {to_text(resp.content) if resp is not None else ''}")

await mcp_host("2의 10제곱을 계산하고 서울 날씨도 알려줘. 지금 몇 시인지도.")

[질문] 2의 10제곱을 계산하고 서울 날씨도 알려줘. 지금 몇 시인지도.


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024


[Host→LLM 결정] tool_calls: [('calculator', {'expression': '2 ** 10'})]
[Client→Server] calculator({'expression': '2 ** 10'}) → 2 ** 10 = 1024



[최종 답변] ;;


---
## 4. 정리

| 계층 | 구현 | 핵심 API |
|---|---|---|
| **Server** | `FastMCP(name=...)` + `@mcp.tool` | 함수 타입힌트·docstring → 도구 스키마 자동 생성 |
| **Client** | `Client(mcp)` (인메모리) | `await list_tools()`, `await call_tool(name, args)` |
| **Host** | Ollama `qwen3:8b` | 도구 스키마를 `bind_tools` → `tool_calls` → Client 로 실행 |

### 인메모리 vs 실제 전송
이 노트북은 학습·테스트를 위해 **인메모리 전송**(서버 객체 직접 전달)을 썼습니다. 실제 배포에서는
서버를 독립 프로세스로 띄우고 표준 전송으로 접속합니다.

```python
# 서버를 stdio 로 실행 (별도 프로세스; 예: server.py)
if __name__ == '__main__':
    mcp.run()                       # 기본 stdio 전송

# 클라이언트에서 접속(전송 문자열/설정으로 자동 감지)
# async with Client('server.py') as client:      # stdio
# async with Client('http://localhost:8000/mcp') as client:   # HTTP
```

- **Server/Client/Host 분리** 덕분에, 같은 Host(LLM)가 여러 MCP 서버(사내 DB·파일·검색 등)를 표준 방식으로 조합할 수 있습니다.

### 참고
- FastMCP: https://gofastmcp.com/getting-started/welcome
- MCP 사양: https://modelcontextprotocol.io